# 🎛️ ComfyCluster: Controller

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlamandaCathartica/ComfyCluster/blob/main/Controller.ipynb)

This is the **only tab you use**. It runs ComfyUI with the Distributed extension and a small GUI around it. The workers (see `Worker.ipynb`, one per Colab account) only render.

**Steps:** start the 4 workers first and copy each pairing code. Paste all the codes into `WORKER_CODES` below (separated by spaces), pick a **GPU runtime**, run the cell and open the private link it prints.

The link contains a secret key; don't share it. For gated models add a Colab secret named `HF_TOKEN`. Models come from `models.json` in the repository, so the controller and the workers always get the same ones.

In [ ]:
#@title ▶️ Start ComfyCluster controller
import json, os, secrets, subprocess, sys, time, urllib.request

REPO_URL = 'https://github.com/AlamandaCathartica/ComfyCluster' #@param {type:"string"}
BRANCH = 'main' #@param {type:"string"}
WORKER_CODES = '' #@param {type:"string"}
USE_DRIVE = False #@param {type:"boolean"}

ROOT = '/content'
REPO_DIR = f'{ROOT}/ComfyCluster'
ENGINE = f'{REPO_DIR}/engine/controller.py'
PORT = 7860
URL_FILE = f'{ROOT}/comfycluster_url.txt'
LOG_FILE = f'{ROOT}/comfycluster_controller.log'
CACHE_DIR = f'{ROOT}/_comfycluster_cache'
KEY = secrets.token_urlsafe(16)

if USE_DRIVE:
  from google.colab import drive
  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
  CACHE_DIR = '/content/drive/MyDrive/ComfyCluster/_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- download / update the code from GitHub ----
subprocess.run(['pkill', '-f', 'engine/controller.py'], check=False)
subprocess.run(['pkill', '-f', 'comfycluster_gui.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
  r = subprocess.run(f'git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD',
                     shell=True, capture_output=True, text=True)
else:
  subprocess.run(['rm', '-rf', REPO_DIR], check=False)
  r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO_DIR], capture_output=True, text=True)
if r.returncode != 0:
  print(r.stdout, r.stderr)
  if not os.path.exists(ENGINE):
    raise SystemExit(f'Could not download {REPO_URL} (branch {BRANCH}). Check the URL/branch and that the repo is public.')
  print('Update failed; using the copy already in this session.')
VERSION = open(f'{REPO_DIR}/VERSION').read().strip()
print(f'ComfyCluster v{VERSION} ready')

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], capture_output=True, text=True)
if r.returncode != 0:
  print(r.stdout[-2000:], r.stderr[-2000:]); raise SystemExit('pip install failed.')

# ---- start the controller (installs ComfyUI, downloads models, pairs workers, opens the GUI) ----
env = dict(os.environ)
try:
  from google.colab import userdata
  env['HF_TOKEN'] = userdata.get('HF_TOKEN')    # optional secret, only needed for gated models
except Exception:
  pass
if os.path.exists(URL_FILE): os.remove(URL_FILE)
cmd = [sys.executable, ENGINE, '--root', ROOT, '--cache-dir', CACHE_DIR, '--models', f'{REPO_DIR}/models.json',
       '--codes', WORKER_CODES, '--key', KEY, '--gui-port', str(PORT), '--url-file', URL_FILE]
proc = subprocess.Popen(cmd, stdout=open(LOG_FILE, 'w'), stderr=subprocess.STDOUT, env=env, start_new_session=True)

def tail(n=1):
  try: return open(LOG_FILE).read().strip().splitlines()[-n:]
  except OSError: return []

from IPython.display import clear_output
shown = ''
print('Installing ComfyUI and starting the controller (the first run takes several minutes)...')
while not os.path.exists(URL_FILE):
  if proc.poll() is not None:
    print(open(LOG_FILE).read()[-3000:]); raise SystemExit('The controller stopped. See the log above.')
  last = (tail() or [''])[0]
  if last != shown: shown = last; print(last)
  time.sleep(3)
LINK = open(URL_FILE).read()
clear_output(wait=True)
print('\n✅ Open ComfyCluster here (keep this link private, it contains an access key):\n')
print(LINK)

# ---- keep this cell running ----
def _status():
  try:
    d = json.load(urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/state?key={KEY}', timeout=5))
    on = sum(1 for w in d['workers'] if w['online'])
    return f"ComfyUI {'up' if d['comfy_ready'] else 'starting'} | workers online {on}/{len(d['workers'])}"
  except Exception as e:
    return f'GUI not responding ({e})'
try:
  while proc.poll() is None:
    clear_output(wait=True)
    print('ComfyCluster link (keep private):\n' + LINK)
    print('\n' + time.strftime('%H:%M:%S'), '-', _status())
    print('\nLeave this cell running. Everything else happens in the GUI.')
    time.sleep(20)
  print(f'The controller exited. See {LOG_FILE} and {ROOT}/comfyui.log')
except KeyboardInterrupt:
  print('Cell stopped. The controller keeps running in the background.')

Closing the browser tab doesn't stop anything. Logs: `/content/comfycluster_controller.log` and `/content/comfyui.log`.